# NIFTY Gap Strategy — v5 (Walk-Forward Signal Backtest)

**Key improvements over v4.3:**
- **Target = PUT option TP hit** (not NIFTY direction) — directly tied to real P&L
- **Clean train/test split**: combos selected on 2024–2025 data ONLY; OOS on 2026
- **Bonferroni correction**: controls false discovery across all tested signal combinations
- **SGX signal**: ^N225 trade-day Open vs D-1 Close (identical to live v4.3 cron)

**Workflow:**
1. Simulate **ALL** 2024–2025 tradeable days → record actual trade outcomes (TP hit / SL / exit)
2. Grid-search signal combos (depth 1–3) → find which correlate with TP hits (Bonferroni-corrected)
3. Apply selected combos to 2026 (OOS) with compounding capital

| Parameter | Value |
|-----------|-------|
| SL | −15% of entry premium |
| TP | +40% of entry premium |
| Hard exit | 11:15 AM IST |
| Strike | ATM − 50 (1-OTM PUT) |
| Training period | 2024 – 2025 |
| OOS period | 2026+ |


In [10]:
import pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date, timedelta
from itertools import combinations
from math import comb as _comb
from scipy.stats import binomtest
import yfinance as yf

warnings.filterwarnings('ignore')

# ── Trade parameters (adjust here) ───────────────────────────────────────────
SL_PCT           = 0.15
TP_PCT           = 0.40
LOT_SIZE         = 75
STRIKE_STEP      = 50
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0
GAP_THR          = 0.0015
GAP_LARGE        = 0.0050
VIX_RISING_THR   = 0.03
VIX_SPIKE_THR    = 0.05
MAX_STALE_DAYS   = 5

# ── Walk-forward split ────────────────────────────────────────────────────────
TRAIN_END  = date(2025, 12, 31)   # combos selected on actual trade outcomes up to this date
OOS_START  = date(2026, 1, 1)     # OOS test period starts here

# ── Combo selection ───────────────────────────────────────────────────────────
MIN_COMBO_N     = 15    # minimum days a combo fires in training period to consider
MAX_COMBO_DEPTH = 4     # 1=singles, 2=pairs, 3=triplets, 4=quadruplets
TOP_N           = 10    # max combos to carry into OOS test
ALPHA           = 0.20  # raised so threshold stays comparable despite more combos
                        # C(13,1..4) = 13+78+286+715 = 1092 total combos
                        # Bonferroni threshold = 0.20/1092 = 0.000183

OLD_CUTOFF = date(2024, 11, 1)
_MON = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

print('Config loaded.')
print(f'Train: up to {TRAIN_END}  |  OOS: {OOS_START}+')
print(f'SL={SL_PCT:.0%}  TP={TP_PCT:.0%}')
_preview = sum(_comb(13, k) for k in range(1, MAX_COMBO_DEPTH + 1))
print(f'Depth: 1–{MAX_COMBO_DEPTH}  |  Total combos: {_preview}  |  Bonferroni threshold: {ALPHA}/{_preview} = {ALPHA/_preview:.6f}')


Config loaded.
Train: up to 2025-12-31  |  OOS: 2026-01-01+
SL=15%  TP=40%
Depth: 1–4  |  Total combos: 1092  |  Bonferroni threshold: 0.2/1092 = 0.000183


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING     = Path.cwd().parent
MARKET_RESEARCH = Path.cwd().parent.parent
ALIGNED_CSV     = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'
MINUTE_CACHE    = GAP_TRADING / 'kite_minute_cache'
MERGE_OLD       = MARKET_RESEARCH / 'merged' / 'old_format'
MERGE_NEW       = MARKET_RESEARCH / 'merged' / 'expiry_wise'

for lbl, p in [('Aligned CSV', ALIGNED_CSV), ('Minute cache', MINUTE_CACHE),
               ('Merge old', MERGE_OLD), ('Merge new', MERGE_NEW)]:
    print(f'{lbl:<14}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Load aligned dataset ──────────────────────────────────────────────────────
aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned = aligned.sort_values('india_date').reset_index(drop=True)
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()
print(f'\nAligned: {aligned.india_date.min().date()} to {aligned.india_date.max().date()}  ({len(aligned)} rows)')

# ── Build spot map: date → NIFTY spot at 09:25 ───────────────────────────────
spot_map = {}
for pkl_path in sorted(MINUTE_CACHE.glob('minute_256265_*.pkl')):
    with open(pkl_path, 'rb') as f:
        chunk = pickle.load(f)
    chunk.index = pd.to_datetime(chunk.index)
    if chunk.index.tzinfo is not None:
        chunk.index = chunk.index.tz_localize(None)
    for dt, row in chunk.iterrows():
        if dt.strftime('%H:%M') == '09:25':
            spot_map[dt.date()] = float(row['open'])
print(f'Spot map: {len(spot_map)} dates with 09:25 NIFTY level')

# ── Fetch ^N225 and compute v5 SGX signal ────────────────────────────────────
ds_start = aligned['india_date'].min().date() - timedelta(days=10)
ds_end   = aligned['india_date'].max().date() + timedelta(days=2)
print(f'\nFetching ^N225 {ds_start} to {ds_end} ...')
try:
    n225 = yf.download('^N225', start=str(ds_start), end=str(ds_end),
                       progress=False, auto_adjust=True)
    if isinstance(n225.columns, pd.MultiIndex):
        n225.columns = n225.columns.get_level_values(0)
    n225.index = pd.to_datetime(n225.index)
    if n225.index.tzinfo is None:
        n225.index = n225.index.tz_localize('UTC')
    n225_ok = len(n225) > 0
    print(f'N225: {len(n225)} rows  ({n225.index[0].date()} to {n225.index[-1].date()})')
except Exception as e:
    n225 = pd.DataFrame(); n225_ok = False
    print(f'WARNING: ^N225 fetch failed: {e}')

def get_n225_sgx_ret(india_date: date):
    if not n225_ok or len(n225) == 0:
        return None
    past_rows  = n225[n225.index.date < india_date]
    today_rows = n225[n225.index.date == india_date]
    if len(past_rows) < 2:
        return None
    close_prev = float(past_rows['Close'].iloc[-1])
    prev_date  = past_rows.index[-1].date()
    if (india_date - prev_date).days > MAX_STALE_DAYS:
        return None
    if len(today_rows) > 0:
        return (float(today_rows['Open'].iloc[0]) - close_prev) / close_prev
    else:
        close_d2 = float(past_rows['Close'].iloc[-2])
        return (close_prev - close_d2) / close_d2

aligned['sgx_ret_v5'] = aligned['india_date'].dt.date.map(get_n225_sgx_ret)
print(f'SGX signal: {aligned["sgx_ret_v5"].notna().sum()}/{len(aligned)} dates have valid N225 data')


Aligned CSV   : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)
Minute cache  : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\kite_minute_cache)
Merge old     : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\merged\old_format)
Merge new     : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\merged\expiry_wise)

Aligned: 2023-03-31 to 2026-04-02  (740 rows)
Spot map: 742 dates with 09:25 NIFTY level

Fetching ^N225 2023-03-21 to 2026-04-04 ...
N225: 743 rows  (2023-03-22 to 2026-04-03)
SGX signal: 736/740 dates have valid N225 data


In [3]:
def d2dmy(d: date) -> str:
    return f'{d.day:02d}{_MON[d.month-1]}{str(d.year)[2:]}'


def _load_old(trade_date: date, expiry_date: date) -> pd.DataFrame | None:
    mon = _MON[trade_date.month - 1]
    fp  = MERGE_OLD / f'2024{mon}' / f'NIFTY-{d2dmy(expiry_date)}-{d2dmy(trade_date)}.csv'
    if not fp.exists():
        return None
    df = pd.read_csv(fp)
    df.columns = [c.strip() for c in df.columns]
    df['time_str'] = df['datetime'].astype(str).str[:5]
    return df


def _load_new(trade_date: date, expiry_date: date) -> pd.DataFrame | None:
    exp_dir = MERGE_NEW / expiry_date.strftime('%Y-%m-%d')
    if not exp_dir.exists():
        return None
    exp_str = f'{expiry_date.day:02d}_{_MON[expiry_date.month-1]}_{str(expiry_date.year)[2:]}'
    files   = list(exp_dir.glob(f'NIFTY_*_*_{exp_str}.csv'))
    if not files:
        return None
    dfs = []
    for fp in files:
        parts = fp.stem.split('_')
        try:
            strike = int(parts[1]); right = parts[2]
        except (ValueError, IndexError):
            continue
        try:
            df = pd.read_csv(fp, usecols=['timestamp','open','high','low','close','volume','oi'])
        except Exception:
            continue
        df['ts'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
        day = df[df['ts'].dt.date == trade_date].copy()
        if day.empty:
            continue
        day['time_str']     = day['ts'].dt.strftime('%H:%M')
        day['strike_price'] = strike
        day['right']        = right
        day.rename(columns={'oi': 'open_interest'}, inplace=True)
        dfs.append(day[['time_str','strike_price','right','open','high','low','close']])
    if not dfs:
        return None
    return (pd.concat(dfs)
              .sort_values(['time_str','strike_price','right'])
              .reset_index(drop=True))


_opt_cache: dict = {}

def load_opt(trade_date: date, expiry_date: date) -> pd.DataFrame | None:
    k = (trade_date, expiry_date)
    if k not in _opt_cache:
        _opt_cache[k] = (_load_old(trade_date, expiry_date)
                         if expiry_date < OLD_CUTOFF
                         else _load_new(trade_date, expiry_date))
    return _opt_cache[k]


print('Option loaders ready.')


Option loaders ready.


In [4]:
# ── NSE helpers ───────────────────────────────────────────────────────────────
NSE_HOLIDAYS = {
    date(2024,  1, 22), date(2024,  3, 25), date(2024,  3, 29), date(2024,  4, 14),
    date(2024,  4, 17), date(2024,  5, 23), date(2024,  6, 17), date(2024,  7, 17),
    date(2024,  8, 15), date(2024, 10,  2), date(2024, 10, 24), date(2024, 11,  1),
    date(2024, 11, 15), date(2024, 12, 25),
    date(2025,  2, 26), date(2025,  3, 14), date(2025,  3, 31), date(2025,  4, 10),
    date(2025,  4, 14), date(2025,  4, 18), date(2025,  5,  1), date(2025,  8, 15),
    date(2025,  8, 27), date(2025, 10,  2), date(2025, 10, 21), date(2025, 10, 22),
    date(2025, 11,  5), date(2025, 12, 25),
    date(2026,  1, 26), date(2026,  3, 26),
}
EVENT_DAYS = {
    date(2024,  2,  1), date(2024,  2,  8), date(2024,  4,  5),
    date(2024,  6,  7), date(2024,  8,  8), date(2024, 10,  9), date(2024, 12,  6),
    date(2025,  2,  1), date(2025,  2,  7), date(2025,  4,  9),
    date(2025,  6,  6), date(2025,  8,  6), date(2025, 10,  8), date(2025, 12,  5),
    date(2026,  2,  1), date(2026,  2,  6), date(2026,  4,  8),
    date(2026,  6,  5), date(2026,  8,  7), date(2026, 10,  7), date(2026, 12,  4),
}
EXPIRY_CHANGE = date(2025, 9, 2)

def is_skip_day(d: date) -> bool:
    return d.weekday() == 0 or d in NSE_HOLIDAYS or d in EVENT_DAYS

def next_expiry(d: date) -> date:
    wd = 1 if d >= EXPIRY_CHANGE else 3
    return d + timedelta(days=(wd - d.weekday()) % 7)

# ── 13 binary signal definitions ─────────────────────────────────────────────
SIGNAL_NAMES = [
    'Gap Up', 'Gap Up Strong', 'Gap Down',
    'Prev India UP', 'Prev India DOWN',
    'US UP', 'US DOWN',
    'SGX UP', 'SGX DOWN',
    'DAX UP',
    'VIX Rising', 'VIX Falling', 'VIX Spike',
]

def compute_signals_v5(row) -> dict:
    gap = float(row['gap_pct']) if pd.notna(row.get('gap_pct', float('nan'))) else 0.0
    def _f(col):
        v = row.get(col)
        return float(v) if pd.notna(v) else None
    sgx  = _f('sgx_ret_v5')
    sp5  = _f('SP500_ret')
    dax  = _f('DAX_ret')
    vix  = _f('VIX_US_ret')
    prev = _f('prev_india_ret')
    return {
        'Gap Up'         : gap >  GAP_THR,
        'Gap Up Strong'  : gap >  GAP_LARGE,
        'Gap Down'       : gap < -GAP_THR,
        'Prev India UP'  : prev is not None and prev > 0,
        'Prev India DOWN': prev is not None and prev < 0,
        'US UP'          : sp5  is not None and sp5  > 0,
        'US DOWN'        : sp5  is not None and sp5  < 0,
        'SGX UP'         : sgx  is not None and sgx  > 0,
        'SGX DOWN'       : sgx  is not None and sgx  < 0,
        'DAX UP'         : dax  is not None and dax  > 0,
        'VIX Rising'     : vix  is not None and vix  > VIX_RISING_THR,
        'VIX Falling'    : vix  is not None and vix  < 0,
        'VIX Spike'      : vix  is not None and vix  > VIX_SPIKE_THR,
    }

def combo_fires(combo_str: str, signals: dict) -> bool:
    return all(signals.get(s.strip(), False) for s in combo_str.split('+'))

# ── Trade simulation ──────────────────────────────────────────────────────────
def round_trip_charges(entry_prem: float, exit_prem: float, lots: int) -> float:
    buy_val  = entry_prem * lots * LOT_SIZE
    sell_val = exit_prem  * lots * LOT_SIZE
    brok  = 20.0 * 2
    stamp = 0.00003  * buy_val
    stt   = 0.000625 * sell_val
    exch  = 0.00053  * (buy_val + sell_val)
    sebi  = 0.000001 * (buy_val + sell_val)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)


def simulate_trade_real(d: date) -> dict | None:
    spot_925 = spot_map.get(d)
    if spot_925 is None:
        return None
    atm    = round(spot_925 / STRIKE_STEP) * STRIKE_STEP
    strike = atm - STRIKE_STEP
    expiry = next_expiry(d)
    dte    = (expiry - d).days
    opt_df = load_opt(d, expiry)
    if opt_df is None:
        return None
    pe = opt_df[(opt_df['strike_price'] == strike) & (opt_df['right'] == 'PE')].copy()
    if pe.empty:
        pe = opt_df[(opt_df['strike_price'] == atm) & (opt_df['right'] == 'PE')].copy()
        if pe.empty:
            return None
        strike = atm
    pe = pe.sort_values('time_str').reset_index(drop=True)
    entry_row = pe[pe['time_str'] == '09:25']
    if entry_row.empty:
        return None
    entry_prem = float(entry_row.iloc[0]['open'])
    if entry_prem < 0.5:
        return None
    sl_px = entry_prem * (1 - SL_PCT)
    tp_px = entry_prem * (1 + TP_PCT)
    monitor = pe[(pe['time_str'] >= '09:26') & (pe['time_str'] <= '11:15')]
    exit_prem, exit_reason, exit_time = None, '11:15 exit', '11:15'
    for _, row in monitor.iterrows():
        lo, hi, t = float(row['low']), float(row['high']), row['time_str']
        if lo <= sl_px:
            exit_prem, exit_reason, exit_time = sl_px, 'Stop Loss', t
            break
        if hi >= tp_px:
            exit_prem, exit_reason, exit_time = tp_px, 'Target Hit', t
            break
    if exit_prem is None:
        exit_row = pe[pe['time_str'] == '11:15']
        if not exit_row.empty:
            exit_prem = float(exit_row.iloc[0]['close'])
        elif not monitor.empty:
            exit_prem = float(monitor.iloc[-1]['close'])
        else:
            exit_prem = entry_prem
    return {
        'expiry': expiry, 'dte': dte, 'atm': int(atm), 'strike': int(strike),
        'entry_prem': round(entry_prem, 2), 'exit_prem': round(exit_prem, 2),
        'pnl_pts': round(exit_prem - entry_prem, 2),
        'exit_reason': exit_reason, 'exit_time': exit_time,
    }

print('NSE helpers, signals, and simulation ready.')


NSE helpers, signals, and simulation ready.


In [5]:
print(f'Simulating ALL tradeable days up to {TRAIN_END} (no signal pre-filter)...')
print(f'This builds the ground truth: for each day, did the PUT option hit TP?')
print()

train_rows    = []
no_data_count = 0

train_aligned = aligned[aligned['india_date'].dt.date <= TRAIN_END]

for _, row in train_aligned.iterrows():
    d = row['india_date'].date()
    if is_skip_day(d):
        continue
    sigs = compute_signals_v5(row)
    res  = simulate_trade_real(d)
    if res is None:
        no_data_count += 1
        continue
    train_row = {
        'date'       : d,
        'win'        : res['exit_reason'] == 'Target Hit',
        'exit_reason': res['exit_reason'],
        'entry_prem' : res['entry_prem'],
    }
    train_row.update(sigs)
    train_rows.append(train_row)

train_df      = pd.DataFrame(train_rows)
base_win_rate = train_df['win'].mean()

print(f'Train period  : {train_aligned["india_date"].min().date()} to {train_aligned["india_date"].max().date()}')
print(f'Tradeable days simulated    : {len(train_df)}')
print(f'  Missing option data skipped : {no_data_count}')
print(f'  Target Hit (wins)           : {int(train_df["win"].sum())}')
print(f'  Stop Loss                   : {(train_df["exit_reason"]=="Stop Loss").sum()}')
print(f'  11:15 exit                  : {(train_df["exit_reason"]=="11:15 exit").sum()}')
print(f'\n  Base win rate (all days)    : {base_win_rate:.1%}')
print(f'\nThis is the baseline — any selected combo must exceed this to have edge.')


Simulating ALL tradeable days up to 2025-12-31 (no signal pre-filter)...
This builds the ground truth: for each day, did the PUT option hit TP?

Train period  : 2023-03-31 to 2025-12-31
Tradeable days simulated    : 361
  Missing option data skipped : 167
  Target Hit (wins)           : 87
  Stop Loss                   : 255
  11:15 exit                  : 19

  Base win rate (all days)    : 24.1%

This is the baseline — any selected combo must exceed this to have edge.


In [11]:
total_combos         = sum(_comb(len(SIGNAL_NAMES), k) for k in range(1, MAX_COMBO_DEPTH + 1))
bonferroni_threshold = ALPHA / total_combos

print(f'Signal universe        : {len(SIGNAL_NAMES)} signals')
print(f'Total combos tested    : {total_combos}  (depth 1 to {MAX_COMBO_DEPTH})')
for d in range(1, MAX_COMBO_DEPTH + 1):
    labels = ['Singles','Pairs','Triplets','Quadruplets']
    print(f'  {labels[d-1]:<12} (depth {d}) : {_comb(len(SIGNAL_NAMES), d)}')
print(f'Base win rate          : {base_win_rate:.1%}')
print(f'Bonferroni threshold   : {ALPHA} / {total_combos} = {bonferroni_threshold:.6f}')
print(f'Min N per combo        : {MIN_COMBO_N}')
print()

combo_results = []
for depth in range(1, MAX_COMBO_DEPTH + 1):
    for combo in combinations(SIGNAL_NAMES, depth):
        mask = train_df[list(combo)].all(axis=1)
        sub  = train_df[mask]
        n    = len(sub)
        if n < MIN_COMBO_N:
            continue
        k     = int(sub['win'].sum())
        p_win = k / n
        edge  = p_win - base_win_rate
        pval  = binomtest(k, n, base_win_rate, alternative='greater').pvalue
        combo_results.append({
            'Signal'     : ' + '.join(combo),
            'Depth'      : depth,
            'N'          : n,
            'Wins'       : k,
            'P_Win%'     : round(p_win * 100, 1),
            'Edge_pp'    : round(edge * 100, 1),
            'p_val'      : round(pval, 6),
            'Bonferroni' : 'PASS' if pval < bonferroni_threshold else '',
        })

results_df = (pd.DataFrame(combo_results)
              .sort_values(['P_Win%', 'N'], ascending=[False, False])
              .reset_index(drop=True))

n_pass = (results_df['Bonferroni'] == 'PASS').sum()
print(f'Combos meeting MIN_N={MIN_COMBO_N} : {len(results_df)}')
print(f'Bonferroni passing          : {n_pass}')
print()
pd.set_option('display.max_rows', 400)
pd.set_option('display.width', 180)
print(results_df.head(50).to_string(index=False))
if n_pass:
    print('\n--- Bonferroni PASS combos ---')
    print(results_df[results_df['Bonferroni'] == 'PASS'].to_string(index=False))


Signal universe        : 13 signals
Total combos tested    : 1092  (depth 1 to 4)
  Singles      (depth 1) : 13
  Pairs        (depth 2) : 78
  Triplets     (depth 3) : 286
  Quadruplets  (depth 4) : 715
Base win rate          : 24.1%
Bonferroni threshold   : 0.2 / 1092 = 0.000183
Min N per combo        : 15

Combos meeting MIN_N=15 : 269
Bonferroni passing          : 0

                                         Signal  Depth  N  Wins  P_Win%  Edge_pp    p_val Bonferroni
                Gap Up + SGX DOWN + VIX Falling      3 18    10    55.6     31.5 0.004079           
                      Gap Up + US UP + SGX DOWN      3 16     8    50.0     25.9 0.021895           
      Gap Up + Prev India DOWN + US UP + DAX UP      4 24    11    45.8     21.7 0.016284           
        US UP + SGX DOWN + DAX UP + VIX Falling      4 20     9    45.0     20.9 0.032821           
                     Gap Up + SGX DOWN + DAX UP      3 16     7    43.8     19.7 0.067049           
      Prev India UP 

In [12]:
# ── Primary: Bonferroni-corrected combos with positive edge ──────────────────
selected = results_df[(results_df['Bonferroni'] == 'PASS') & (results_df['Edge_pp'] > 0)].copy()
selection_method = f'Bonferroni-corrected  (p < {bonferroni_threshold:.6f})'

# ── Fallback: if < 5 pass Bonferroni, use uncorrected top combos ─────────────
if len(selected) < 5:
    selected = (results_df[(results_df['p_val'] < ALPHA) & (results_df['Edge_pp'] > 0)]
                .nlargest(TOP_N, 'Edge_pp')
                .copy())
    selection_method = (f'UNCORRECTED fallback  (p < {ALPHA}, Bonferroni too strict)  '
                        f'— edge estimates are likely overstated')
    print(f'WARNING: Fewer than 5 combos passed Bonferroni correction.')
    print(f'Falling back to uncorrected top-{TOP_N} by edge. Interpret results cautiously.\n')
else:
    selected = selected.nlargest(TOP_N, 'Edge_pp').copy()

bear_combos_v5 = list(selected['Signal'])

print(f'Selection method : {selection_method}')
print(f'Combos selected  : {len(selected)} (capped at TOP_N={TOP_N})')
print()
print(selected[['Signal','Depth','N','Wins','P_Win%','Edge_pp','p_val','Bonferroni']].to_string(index=False))

selected.to_csv('v5_selected_combos.csv', index=False)
print(f'\nSaved: v5_selected_combos.csv')
print(f'\nSelected combos for OOS:')
for i, c in enumerate(bear_combos_v5, 1):
    print(f'  {i:2d}. {c}')


Falling back to uncorrected top-10 by edge. Interpret results cautiously.

Selection method : UNCORRECTED fallback  (p < 0.2, Bonferroni too strict)  — edge estimates are likely overstated
Combos selected  : 10 (capped at TOP_N=10)

                                         Signal  Depth  N  Wins  P_Win%  Edge_pp    p_val Bonferroni
                Gap Up + SGX DOWN + VIX Falling      3 18    10    55.6     31.5 0.004079           
                      Gap Up + US UP + SGX DOWN      3 16     8    50.0     25.9 0.021895           
      Gap Up + Prev India DOWN + US UP + DAX UP      4 24    11    45.8     21.7 0.016284           
        US UP + SGX DOWN + DAX UP + VIX Falling      4 20     9    45.0     20.9 0.032821           
                     Gap Up + SGX DOWN + DAX UP      3 16     7    43.8     19.7 0.067049           
      Prev India UP + US UP + SGX DOWN + DAX UP      4 16     7    43.8     19.7 0.067049           
                             SGX UP + VIX Spike      2 21   

In [13]:
def run_oos(period_label: str, period_df: pd.DataFrame):
    '''Run OOS backtest for a given subset of aligned data.'''
    results, no_data = [], 0

    for _, row in period_df.iterrows():
        d = row['india_date'].date()
        if is_skip_day(d):
            continue
        sigs  = compute_signals_v5(row)
        fired = [c for c in bear_combos_v5 if combo_fires(c, sigs)]
        if not fired:
            continue
        res = simulate_trade_real(d)
        if res is None:
            no_data += 1
            continue
        results.append({
            'Date'       : d,
            'Combo'      : fired[0],
            'DTE'        : res['dte'],
            'Entry (pts)': res['entry_prem'],
            'Exit (pts)' : res['exit_prem'],
            'PnL (pts)'  : res['pnl_pts'],
            'Exit Reason': res['exit_reason'],
            'Exit Time'  : res['exit_time'],
        })

    if not results:
        print(f'{period_label}: no trades fired.')
        return pd.DataFrame(), pd.DataFrame()

    res_df = pd.DataFrame(results)

    # Compounding capital tracker
    capital, peak, ledger_rows = STARTING_CAPITAL, STARTING_CAPITAL, []
    for _, row in res_df.iterrows():
        ep, xp   = float(row['Entry (pts)']), float(row['Exit (pts)'])
        pnl_pts  = float(row['PnL (pts)'])
        dte      = int(row['DTE'])
        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0:
            continue
        lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0:
            lots = min(lots, DTE0_MAX_LOTS)
        charges   = round_trip_charges(ep, xp, lots)
        trade_pnl = pnl_pts * LOT_SIZE * lots - charges
        capital  += trade_pnl
        peak      = max(peak, capital)
        ledger_rows.append({
            'Trade#'     : len(ledger_rows) + 1,
            'Date'       : row['Date'],
            'DTE'        : dte,
            'Lots'       : lots,
            'Entry (pts)': ep,
            'Exit (pts)' : xp,
            'PnL (pts)'  : pnl_pts,
            'Charges'    : charges,
            'Trade PnL'  : round(trade_pnl, 2),
            'Capital'    : round(capital, 2),
            'Drawdown%'  : round((peak - capital) / peak * 100, 2),
            'Exit Reason': row['Exit Reason'],
        })

    ledger   = pd.DataFrame(ledger_rows)
    wins     = (ledger['Trade PnL'] > 0).sum()
    total    = len(ledger)
    roi      = (capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    max_dd   = ledger['Drawdown%'].max()
    avg_win  = ledger.loc[ledger['Trade PnL'] > 0,  'Trade PnL'].mean() if wins > 0            else 0.0
    avg_loss = ledger.loc[ledger['Trade PnL'] <= 0, 'Trade PnL'].mean() if wins < total else 0.0

    print(f'\n{"="*55}')
    print(f'  {period_label}')
    print(f'{"="*55}')
    print(f'  Signal days (combo fired) : {len(res_df) + no_data}')
    print(f'  Traded                    : {total}  (skipped {no_data} — missing data)')
    print(f'  Win rate                  : {wins/total*100:.1f}%'
          f'  (base train all-days: {base_win_rate:.1%})')
    print(f'  ROI                       : {roi:+.1f}%')
    print(f'  Max drawdown              : {max_dd:.1f}%')
    print(f'  Avg win / loss            : Rs {avg_win:,.0f} / Rs {avg_loss:,.0f}')
    print(f'{"="*55}')
    print(ledger['Exit Reason'].value_counts().to_string())
    print()
    print(ledger[['Trade#','Date','DTE','Lots','Entry (pts)','Exit (pts)',
                  'PnL (pts)','Trade PnL','Capital','Drawdown%','Exit Reason']].to_string(index=False))

    return ledger, res_df


# ── OOS 2026+ ─────────────────────────────────────────────────────────────────
df_oos = aligned[aligned['india_date'].dt.date >= OOS_START]
ledger_oos, res_oos = run_oos(f'OOS {OOS_START.year}+ (out-of-sample)', df_oos)



  OOS 2026+ (out-of-sample)
  Signal days (combo fired) : 10
  Traded                    : 8  (skipped 2 — missing data)
  Win rate                  : 12.5%  (base train all-days: 24.1%)
  ROI                       : -37.9%
  Max drawdown              : 37.9%
  Avg win / loss            : Rs 33,678 / Rs -15,631
Exit Reason
Stop Loss     4
11:15 exit    3
Target Hit    1

 Trade#       Date  DTE  Lots  Entry (pts)  Exit (pts)  PnL (pts)  Trade PnL   Capital  Drawdown% Exit Reason
      1 2026-01-07    6    25        73.50       62.48     -11.02  -20946.81 179053.19      10.47   Stop Loss
      2 2026-01-14    6    23       101.20       86.02     -15.18  -26533.03 152520.16      23.74   Stop Loss
      3 2026-01-16    4    25        72.50       61.62     -10.88  -20681.06 131839.10      34.08   Stop Loss
      4 2026-01-28    6     8       199.50      195.05      -4.45   -2942.27 128896.83      35.55  11:15 exit
      5 2026-02-03    0    10       113.10      158.34      45.24   33678.4

In [14]:
# ── In-sample reference: run selected combos on training data ─────────────────
df_train = aligned[aligned['india_date'].dt.date <= TRAIN_END]
ledger_train, _ = run_oos(f'TRAIN 2024–2025 (in-sample reference)', df_train)


def _summarize(ledger, label):
    if ledger.empty:
        return {'Period': label, 'Trades': 0, 'Win%': 'N/A', 'ROI': 'N/A', 'MaxDD': 'N/A'}
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    roi   = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['Drawdown%'].max()
    return {
        'Period': label,
        'Trades': total,
        'Win%'  : f'{wins/total*100:.1f}%',
        'ROI'   : f'{roi:+.1f}%',
        'MaxDD' : f'{maxdd:.1f}%',
    }

rows = [
    _summarize(ledger_train, 'TRAIN 2024–2025  (in-sample)'),
    _summarize(ledger_oos,   f'OOS   {OOS_START.year}+        (out-of-sample)'),
]
summary = pd.DataFrame(rows)

print()
print('=' * 62)
print('  v5 WALK-FORWARD SUMMARY')
print('=' * 62)
print(summary.to_string(index=False))
print('=' * 62)
print('  v43 reference (in-sample combos, Jan 2024–Apr 2026):')
print('  120 trades  |  33.3% win  |  +147.7% ROI  |  34.2% MaxDD')
print('=' * 62)
print()
print(f'  Selection   : {selection_method}')
print(f'  Bonferroni  : {ALPHA} / {total_combos} = {bonferroni_threshold:.6f}')
print(f'  Base train win rate (ALL tradeable days) : {base_win_rate:.1%}')
print(f'  Combos selected : {len(bear_combos_v5)}')
for c in bear_combos_v5:
    r = selected[selected['Signal'] == c].iloc[0]
    print(f'    {c}  '
          f'(train N={int(r["N"])}, P_Win={r["P_Win%"]:.1f}%, Edge={r["Edge_pp"]:+.1f}pp)')



  TRAIN 2024–2025 (in-sample reference)
  Signal days (combo fired) : 130
  Traded                    : 88  (skipped 42 — missing data)
  Win rate                  : 36.4%  (base train all-days: 24.1%)
  ROI                       : +291.8%
  Max drawdown              : 36.0%
  Avg win / loss            : Rs 53,750 / Rs -20,293
Exit Reason
Stop Loss     54
Target Hit    31
11:15 exit     3

 Trade#       Date  DTE  Lots  Entry (pts)  Exit (pts)  PnL (pts)  Trade PnL   Capital  Drawdown% Exit Reason
      1 2024-01-09    2    25        55.40       77.56      22.16   41252.59 241252.59       0.00  Target Hit
      2 2024-01-17    1    25        71.70       60.95     -10.76  -20453.50 220799.09       8.48   Stop Loss
      3 2024-01-19    6    23       124.50      105.83     -18.67  -32622.44 188176.65      22.00   Stop Loss
      4 2024-01-25    0    10        63.25       53.76      -9.49   -7246.31 180930.34      25.00   Stop Loss
      5 2024-02-06    2    18       129.85      110.37  